# D* Lite Concept Demo

This notebook is a self-learning walkthrough of repeated replanning in a changing environment.

Learning goals:
- understand why repeated replanning is needed during execution,
- measure search effort in a dynamic loop,
- connect this baseline to what D* Lite improves in practice.

For clarity, this notebook uses repeated A* from the current robot state as a teaching baseline.

In [ ]:
%matplotlib inline

import heapq
import math

import matplotlib.pyplot as plt
import numpy as np

## 1. Build a dynamic grid

We create a map with corridors and update points so replanning behavior is visible.

In [ ]:
H, W = 40, 40
grid = np.zeros((H, W), dtype=np.uint8)

# Initial obstacles
grid[8:32, 12] = 1
grid[8:32, 28] = 1
grid[20, 12:29] = 1

# Keep several passages open
grid[12, 12] = 0
grid[30, 28] = 0
grid[20, 20] = 0

start = (2, 2)
goal = (37, 37)

## 2. Search utilities

These helpers implement a baseline A* used repeatedly after each map update.

Key concept: D* Lite avoids full restart by reusing search structure.

In [ ]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c, h, w):
    return 0 <= r < h and 0 <= c < w

def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def reconstruct(parent, node):
    path = [node]
    while node in parent:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path

def astar(grid_map, s, g):
    h, w = grid_map.shape
    open_heap = []
    parent = {}
    g_cost = {s: 0.0}
    closed = set()

    heapq.heappush(open_heap, (heuristic(s, g), 0.0, s))
    expansions = 0

    while open_heap:
        _, curr_g, current = heapq.heappop(open_heap)
        if current in closed:
            continue

        closed.add(current)
        expansions += 1

        if current == g:
            return reconstruct(parent, current), g_cost[current], expansions

        for dr, dc in DIRS:
            nr, nc = current[0] + dr, current[1] + dc
            if not in_bounds(nr, nc, h, w):
                continue
            if grid_map[nr, nc] == 1:
                continue

            nxt = (nr, nc)
            new_g = curr_g + 1.0
            if new_g < g_cost.get(nxt, math.inf):
                g_cost[nxt] = new_g
                parent[nxt] = current
                f = new_g + heuristic(nxt, g)
                heapq.heappush(open_heap, (f, new_g, nxt))

    return None, math.inf, expansions

## 3. Simulate movement with periodic map updates

The robot advances step by step. At scheduled times, new obstacles are inserted, then replanning is triggered.

In [ ]:
grid_dyn = grid.copy()
robot = start
trajectory = [robot]
replans = []
total_expansions = 0

# Timed updates to mimic newly observed obstacles
scheduled_updates = {
    6: [(20, 20), (21, 20), (22, 20)],
    14: [(30, 28), (30, 27), (30, 26)]
}

max_steps = 60

for t in range(max_steps):
    if robot == goal:
        break

    if t in scheduled_updates:
        for r, c in scheduled_updates[t]:
            if (r, c) != robot and (r, c) != goal:
                grid_dyn[r, c] = 1

    path, cost, exp = astar(grid_dyn, robot, goal)
    total_expansions += exp
    replans.append((t, robot, exp, cost))

    if path is None or len(path) < 2:
        print('No feasible route remains.')
        break

    robot = path[1]
    trajectory.append(robot)

replans[-1] if replans else None

## 4. Visualize trajectory on the updated map

Use this plot to see where updates forced trajectory changes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(grid_dyn, cmap='Greys', origin='upper')

if trajectory:
    xs = [p[1] for p in trajectory]
    ys = [p[0] for p in trajectory]
    ax.plot(xs, ys, color='deepskyblue', linewidth=2.0, label='robot trajectory')

ax.scatter(start[1], start[0], c='limegreen', s=90, marker='o', label='start')
ax.scatter(goal[1], goal[0], c='crimson', s=90, marker='*', label='goal')
ax.scatter(robot[1], robot[0], c='gold', s=80, marker='s', label='final robot state')

ax.set_title('Repeated replanning under map updates')
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc='upper right')
plt.show()

In [ ]:
print(f'Replanning rounds: {len(replans)}')
print(f'Total node expansions (baseline repeated A*): {total_expansions}')
print(f'Goal reached: {robot == goal}')

## 5. Interpretation and exercises

Takeaways:
- Replanning from scratch is simple but can be expensive over many updates.
- D* Lite is designed to repair search efficiently as costs/obstacles change.

Try it yourself:
1. Add another scheduled update and rerun.
2. Compare total expansion count before and after your change.
3. Reduce `max_steps` and observe if goal completion changes.